# CC3104 - Aprendizaje por Refuerzo
## Hoja de Trabajo 1

### Task 1

**1. Elija un sistema real de su área de interés. Identifique y justifique:**
Sistema elegido: **Robot aspiradora (ej. Roomba)**

*   **(a) Estado $S_t$:** Las coordenadas actuales del robot en la habitación, el nivel de batería actual y el estado de sus sensores de colisión o suciedad inmediata.
*   **(b) Espacio de acciones $\mathcal{A}$:** Avanzar, girar a la izquierda, girar a la derecha, detenerse, volver a la base de carga.
*   **(c) Función de recompensa $R$:** 
    *   +10 por limpiar una zona sucia.
    *   -1 por cada paso de tiempo (para incentivar que limpie rápido).
    *   -50 por quedarse sin batería antes de llegar a la base.
*   **(d) Observabilidad:** **Parcialmente observable**. El robot no conoce el mapa completo y exacto de la casa al iniciar, ni sabe dónde está toda la suciedad o si hay obstáculos dinámicos (personas moviéndose). Depende de sus sensores locales para inferir su estado en el entorno.

**2. Para el sistema que eligió, argumente qué valor de $\gamma$ sería más apropiado y por qué. Incluya al menos un ejemplo numérico.**
Un valor de $\gamma$ (factor de descuento) **alto** (por ejemplo, $\gamma = 0.95$) sería más apropiado. El robot aspiradora debe preocuparse por objetivos a largo plazo, como limpiar toda la casa y sobrevivir para volver a su base de carga, no solo por la suciedad inmediata.

*Ejemplo numérico:*
Supongamos que el robot tiene poca batería y morir le da un castigo de -50 en 5 pasos. A su lado tiene una mancha de polvo que le da +5.
*   Con $\gamma = 0.1$ (bajo), valora más lo inmediato: $G \approx 5 + 0.1^5(-50) = 5 - 0.0005 = 4.99$. El robot elige limpiar y luego muere.
*   Con $\gamma = 0.95$ (alto), valora el futuro: $G \approx 5 + 0.95^5(-50) = 5 - 38.68 = -33.68$. El robot se da cuenta de que es una mala idea y preferirá ir a la base (recompensa a largo plazo mejor).

**3. Describa en lenguaje natural una política determinista y una estocástica para su sistema elegido. ¿En qué escenario preferiría una sobre otra?**
*   **Política Determinista $\pi(s)$:** Siempre que el sensor frontal detecte un obstáculo, el robot girará exactamente 90 grados a la derecha. 
*   **Política Estocástica $\pi(a|s)$:** Cuando el sensor frontal detecta un obstáculo, el robot tiene un 50% de probabilidad de girar a la derecha y un 50% de probabilidad de girar a la izquierda.

*Preferencia:* Preferiría una **política estocástica** en escenarios de exploración o cuando el entorno es parcialmente observable (PO). Por ejemplo, si el robot queda atrapado en una esquina compleja, una política determinista podría hacerlo entrar en un bucle infinito (chocar, girar, chocar, girar). Una política estocástica asegura que eventualmente tomará una acción diferente y logrará escapar.

### Task 2

A continuación se presenta el código original modificado paso a paso según las instrucciones.

In [1]:
import numpy as np
import random

GRID_SIZE = 4
GOAL_STATE = (3, 3)
ACTIONS = {
    0: (-1, 0),   # Arriba
    1: (1, 0),    # Abajo
    2: (0, -1),   # Izquierda
    3: (0, 1),    # Derecha
}
ACTION_NAMES = {0: "↑", 1: "↓", 2: "←", 3: "→"}

class GridWorldEnv:
    def __init__(self, trap_states, step_reward):
        self.trap_states = trap_states
        self.step_reward = step_reward
        
    def reset(self):
        self.state = (0, 0)
        self.done = False
        return self.state
    
    def step(self, action):
        if self.done: raise RuntimeError("Episodio terminado")
        
        dr, dc = ACTIONS[action]
        r, c = self.state
        new_r = max(0, min(GRID_SIZE - 1, r + dr))
        new_c = max(0, min(GRID_SIZE - 1, c + dc))
        next_state = (new_r, new_c)
        
        if next_state == GOAL_STATE:
            reward = +10.0
            self.done = True
        elif next_state in self.trap_states:
            reward = -5.0
            self.done = True
        else:
            reward = self.step_reward
            
        self.state = next_state
        return next_state, reward, self.done

class RandomAgent:
    def __init__(self, action_space_size):
        self.n_actions = action_space_size
    def select_action(self, state):
        return random.randint(0, self.n_actions - 1)

def run_episode(env, agent, gamma, max_steps=50, verbose=False):
    state = env.reset()
    trajectory = []
    for t in range(max_steps):
        action = agent.select_action(state)
        next_state, reward, done = env.step(action)
        trajectory.append((state, action, reward, next_state))
        state = next_state
        if done: break
            
    # G_0 = R_1 + gamma*R_2 + gamma^2*R_3 ...
    G = 0.0
    returns = []
    for _, _, r, _ in reversed(trajectory):
        G = r + gamma * G
        returns.insert(0, G)
    return trajectory, returns

def run_experiment(trap_states, step_reward, gamma, n_episodes=5):
    # Fijamos la semilla para que los resultados sean reproducibles
    random.seed(42)
    np.random.seed(42)
    
    env = GridWorldEnv(trap_states=trap_states, step_reward=step_reward)
    agent = RandomAgent(action_space_size=len(ACTIONS))
    
    all_returns = []
    for ep in range(n_episodes):
        _, returns = run_episode(env, agent, gamma)
        all_returns.append(returns[0])
    
    return np.mean(all_returns)

**1. Cambie GAMMA a 0.5 y a 0.99. ¿Cómo cambia el retorno $G_0$ promedio entre episodios?**

In [2]:
# Parámetros originales
trap_states_orig = {(1, 1), (2, 3)}
step_reward_orig = -1.0

# 1. GAMMA = 0.5
gamma_05 = 0.5
ret_05 = run_experiment(trap_states_orig, step_reward_orig, gamma_05, n_episodes=10)
print(f"Retorno G_0 promedio con GAMMA = 0.5: {ret_05:.4f}")

# 1. GAMMA = 0.99
gamma_099 = 0.99
ret_099 = run_experiment(trap_states_orig, step_reward_orig, gamma_099, n_episodes=10)
print(f"Retorno G_0 promedio con GAMMA = 0.99: {ret_099:.4f}")

Retorno G_0 promedio con GAMMA = 0.5: -2.2267
Retorno G_0 promedio con GAMMA = 0.99: -16.2450


*Explicación:*
Al cambiar el factor de descuento $\gamma$, notamos que con $\gamma = 0.5$ el retorno es menos negativo y se acota más rápido que con $\gamma = 0.99$. Esto se debe a que $\gamma = 0.5$ penaliza fuertemente el valor de las recompensas futuras lejanas, restando peso a los costos de los pasos extra a largo plazo. Con $\gamma = 0.99$, el agente da casi igual importancia a lo que ocurra en el futuro que a lo presente, sumando casi por completo los -1 por paso durante todo el episodio, dando un retorno promedio más bajo (más negativo).

**2. Agregue un nuevo TRAP_STATE en la posición (0, 3). ¿Cómo afecta esto la distribución de retornos?**

In [3]:
# Se usa GAMMA original = 0.95 para comparar correctamente el efecto de la trampa
gamma_orig = 0.95

# Sin la nueva trampa
ret_sin_trampa = run_experiment(trap_states_orig, step_reward_orig, gamma_orig, n_episodes=10)
print(f"Retorno promedio SIN nueva trampa: {ret_sin_trampa:.4f}")

# Con la nueva trampa en (0,3)
trap_states_nuevas = {(1, 1), (2, 3), (0, 3)}
ret_con_trampa = run_experiment(trap_states_nuevas, step_reward_orig, gamma_orig, n_episodes=10)
print(f"Retorno promedio CON nueva trampa en (0,3): {ret_con_trampa:.4f}")

Retorno promedio SIN nueva trampa: -11.5229
Retorno promedio CON nueva trampa en (0,3): -10.8628


*Explicación:*
Al agregar un nuevo estado trampa en `(0, 3)`, la distribución de los retornos $G_0$ tiende hacia valores más negativos. Debido a que el agente tiene una política completamente aleatoria, la existencia de una trampa adicional aumenta la probabilidad de que el episodio finalice rápidamente con una recompensa de -5 en vez de poder vagar hasta la meta (+10). Esto baja considerablemente el retorno promedio.

**3. Modifique la recompensa por paso de -1 a -0.1. ¿Qué efecto tiene en el comportamiento del agente?**

In [ ]:
# Modificamos la recompensa por paso a -0.1
step_reward_nuevo = -0.1

ret_recompensa_orig = run_experiment(trap_states_orig, step_reward_orig, gamma_orig, n_episodes=10)
print(f"Retorno G_0 promedio con r = -1.0: {ret_recompensa_orig:.4f}")

ret_recompensa_nueva = run_experiment(trap_states_orig, step_reward_nuevo, gamma_orig, n_episodes=10)
print(f"Retorno G_0 promedio con r = -0.1: {ret_recompensa_nueva:.4f}")

Retorno G_0 promedio con r = -1.0: -11.5229
Retorno G_0 promedio con r = -0.1: -3.5496


*Explicación:*
El comportamiento del *agente aleatorio* (RandomAgent) no cambia porque su política $\pi$ no depende de las recompensas recibidas. Sin embargo, matemáticamente vemos que el retorno $G_0$ aumenta significativamente. Si este agente *pudiera aprender*, una recompensa de -0.1 en lugar de -1.0 haría que el agente tenga menos urgencia por alcanzar la meta. Es decir, estaría dispuesto a tomar trayectorias más largas y exploratorias, reduciendo el riesgo de caer en trampas, a costa de tardar más tiempo.